# min / mean subtraction test

Compare suite2p output on the same recording under three input variants:

1. **baseline** &mdash; raw int16 movie, no global offset
2. **min sub** &mdash; `data - global_min` (matches the LBM-CaImAn-MATLAB step)
3. **mean sub** &mdash; `data - global_mean`

Each variant is materialized to a `.zarr` and run through `lsp.pipeline()` with default ops.

Source: `E:\datasets\lbm\2025-07-27_mk355-kbarber\raw` (two ScanImage tifs, ~11 GB int16).

Outputs land under `C:\Users\Administrator\repos\2026-04-08_mk355_min_sub_test\`.

In [ ]:
import gc
from pathlib import Path

import numpy as np
import mbo_utilities as mbo
import lbm_suite2p_python as lsp

RAW_DIR  = Path(r"E:\datasets\lbm\2025-07-27_mk355-kbarber\raw")
OUT_ROOT = Path(r"C:\Users\Administrator\repos\2026-04-08_mk355_min_sub_test")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_ZARR = OUT_ROOT / "baseline.zarr"
MIN_ZARR      = OUT_ROOT / "min_sub.zarr"
MEAN_ZARR     = OUT_ROOT / "mean_sub.zarr"

BASELINE_RESULTS = OUT_ROOT / "baseline_results"
MIN_RESULTS      = OUT_ROOT / "min_sub_results"
MEAN_RESULTS     = OUT_ROOT / "mean_sub_results"

I16 = np.iinfo(np.int16)

print("raw:", RAW_DIR)
print("out:", OUT_ROOT)

## 1. baseline

Read the raw scanimage tifs as a lazy `LBMArray`, write to zarr, run suite2p with defaults.

In [ ]:
raw = mbo.imread(RAW_DIR)
print(raw)
print("shape:", raw.shape, "dtype:", raw.dtype)

mbo.imwrite(raw, BASELINE_ZARR, ext=".zarr", overwrite=True)

lsp.pipeline(BASELINE_ZARR, save_path=BASELINE_RESULTS)

## 2. global statistics

Stream the raw movie once to get the scalar offsets used by the next two cells. `LBMArray.min()` / `.mean()` are streaming reductions, so this does not load the full array.

In [ ]:
raw = mbo.imread(RAW_DIR)
print("streaming reduction over raw...")
g_min  = int(raw.min())
g_mean = float(raw.mean())
print(f"global min  = {g_min}")
print(f"global mean = {g_mean:.4f}")
print(f"int16 range = [{I16.min}, {I16.max}]")

## 3. min subtracted

Materialize the raw movie (~11 GB int16), subtract the global minimum in-place via int32 chunks (clipped to int16), wrap as a `NumpyArray` carrying the original metadata, write zarr, run suite2p.

In [ ]:
raw = mbo.imread(RAW_DIR)
print(f"materializing {raw.shape} {raw.dtype}...")
data = raw[:]  # full int16 array

print(f"subtracting global min ({g_min}) per timepoint...")
for t in range(data.shape[0]):
    block = data[t].astype(np.int32)
    block -= g_min
    np.clip(block, I16.min, I16.max, out=block)
    data[t] = block.astype(np.int16)

print("new range:", int(data.min()), int(data.max()))

arr_min = mbo.imread(data, metadata=dict(raw.metadata))
mbo.imwrite(arr_min, MIN_ZARR, ext=".zarr", overwrite=True)

del data, arr_min, raw
gc.collect()

lsp.pipeline(MIN_ZARR, save_path=MIN_RESULTS)

## 4. mean subtracted

Same as above, but subtract the rounded global mean. Result is recentered around zero in int16.

In [ ]:
raw = mbo.imread(RAW_DIR)
print(f"materializing {raw.shape} {raw.dtype}...")
data = raw[:]

g_mean_int = int(round(g_mean))
print(f"subtracting global mean ({g_mean_int}) per timepoint...")
for t in range(data.shape[0]):
    block = data[t].astype(np.int32)
    block -= g_mean_int
    np.clip(block, I16.min, I16.max, out=block)
    data[t] = block.astype(np.int16)

print("new range:", int(data.min()), int(data.max()))

arr_mean = mbo.imread(data, metadata=dict(raw.metadata))
mbo.imwrite(arr_mean, MEAN_ZARR, ext=".zarr", overwrite=True)

del data, arr_mean, raw
gc.collect()

lsp.pipeline(MEAN_ZARR, save_path=MEAN_RESULTS)

## 5. quick comparison

ROI counts and accepted-cell counts per condition. Use the per-plane suite2p outputs in each results directory for deeper inspection.

In [ ]:
def summarize(results_dir, label):
    results_dir = Path(results_dir)
    if not results_dir.exists():
        print(f"{label}: <missing>")
        return
    stat_files   = sorted(results_dir.rglob("stat.npy"))
    iscell_files = sorted(results_dir.rglob("iscell.npy"))
    if not stat_files:
        print(f"{label}: no stat.npy")
        return
    n_total = sum(len(np.load(f, allow_pickle=True)) for f in stat_files)
    n_cells = sum(int(np.load(f, allow_pickle=True)[:, 0].sum()) for f in iscell_files)
    print(f"{label}: {len(stat_files)} planes  {n_total:>6} ROIs  {n_cells:>6} accepted")

summarize(BASELINE_RESULTS, "baseline")
summarize(MIN_RESULTS,      "min sub ")
summarize(MEAN_RESULTS,     "mean sub")